In [58]:
import numpy as np
import datetime
import os
from netCDF4 import Dataset
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
np.float = float
import cartopy
from regrid import regrid
import tqdm
import warnings
warnings.filterwarnings("ignore")

In [59]:
datapath_in = '/Users/carmennab/cpom_server/home/cjn/pb_new/freeboard_daily_interpolated/daily_numpys/CS2/'
datapath_out = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2/'

In [60]:
datapath_aux = '/Users/carmennab/Dropbox/pb_new/auxiliary/'
ease_lons = np.load(f'{datapath_aux}lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}lat_cent.npy')

In [62]:
rho_w = 1023.9
rho_i = 916.7 # FYI density everywhere
c = 3.
cs = 2.4

In [63]:
for year in tqdm.trange(2010,2023):

    d1 = datetime.date(year, 1, 1)
    d2 = datetime.date(year, 12, 31)
    dates = [d1 + datetime.timedelta(days=x) for x in range((d2-d1).days + 1)]

    ## create yearly arrays
    rf_uncorrected = np.full((len(dates),ease_lons.shape[0],ease_lons.shape[1]),np.nan)
    sits = np.full((len(dates),ease_lons.shape[0],ease_lons.shape[1]),np.nan)

    for count,date in enumerate(dates):
        str_date = date.strftime('%Y%m%d')

        ### check if file exists
        filepath = f'{datapath_in}FB_interp_{year}_25km_{str_date}.npy'
        if os.path.exists(filepath) == True: 
            rf = np.array(np.load(filepath))
            rf_uncorrected[count] = rf

    ## load auxiliary data
    hs = np.array(Dataset(f'{datapath_aux}snow_depth/MERRA2/{year}.nc')['snow_depth'])
    rho_s = np.array(Dataset(f'{datapath_aux}snow_density/MERRA2/{year}.nc')['snow_density'])

    em_correction = np.array(Dataset(f'{datapath_aux}em_correction/{year}.nc')['em_correction'])
    em_correction[hs>0.6] = np.nan
    sd_redu = (1-hs/0.6)
    sd_redu[sd_redu<0] = 0
    em_correction[np.isnan(em_correction)] = 0

    alpha = np.full(hs.shape,1.0)
    months = np.array([date.month for date in dates])

    summer_mask = np.where((months > 4) & (months < 10))
    alpha[summer_mask] = 0.9

    if (~np.isnan(hs).all()) & (~np.isnan(rf_uncorrected).all()):

        ### calculate sea ice thickness
        rf_combined = rf_uncorrected.copy()
        rf_corrected = rf_uncorrected - (em_correction * sd_redu) # correct summer freeboards
        rf_corrected[rf_corrected>0.7] = np.nan # remove summer freeboards > 0.7m 
        rf_combined[summer_mask] = rf_corrected[summer_mask] ## combine summer corrected & winter uncorrected
        ice_freeboard = rf_combined + (((alpha * (c/cs)) -1) * hs)  # isobel thesis pg 114
        sits = ((ice_freeboard * rho_w) + (hs * rho_s)) / (rho_w - rho_i)
        sits[np.isnan(rf_combined)] = np.nan
        sits[np.isnan(hs)] = np.nan

        if (~np.isnan(sits).all()):

            for count,date in enumerate(dates):
                str_date = date.strftime('%Y%m%d')
                sit = sits[count]
                sd = hs[count]

                if (~np.isnan(sit).all()):
                    filepath_out = f'{datapath_out}MERRA2/{year}/{str_date}.nc'

                    if os.path.exists(filepath_out) == True: 
                        ! rm {filepath_out}
                    data_vars = {'sea_ice_thickness':(['x','y'],sit),
                                'snow_depth':(['x','y'],sd)}
                    ds = xr.Dataset(data_vars = data_vars,
                                    coords={'longitude':(['x','y'],ease_lons),
                                            'latitude':(['x','y'],ease_lats)})
                    ds.to_netcdf(filepath_out,'w')


100%|███████████████████████████████████████████| 13/13 [06:33<00:00, 30.24s/it]
